# Food-11 — Data Preprocessing
Run this notebook **once** before training. It:
1. Extracts the raw zip
2. Resizes all images to 100×100 and saves them to `dataset/food11_100x100/` (persistent NFS)
3. Computes per-channel mean and std on the training set
4. Saves stats to `dataset/stats.json` for `train.ipynb` to load

In [10]:
import os, zipfile, json
import numpy as np
from pathlib import Path
from PIL import Image

import torch
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, Dataset


ROOT = os.path.expanduser('~/QIAO0042/models/ineedfood/')
os.chdir(ROOT)
print('Working dir:', os.getcwd())

Working dir: /scratch-share/QIAO0042/models/ineedfood


## 1 — Extract Dataset

In [2]:
DATASET_ZIP  = 'dataset/food11'
DATASET_ROOT = 'dataset/food11_extracted'

## 2 — Locate Split Directories

In [3]:
def find_split_dir(root, name_candidates):
    for dirpath, dirnames, _ in os.walk(root):
        for d in dirnames:
            if d.lower() in name_candidates:
                return os.path.join(dirpath, d)
    return None

TRAIN_DIR = find_split_dir(DATASET_ROOT, {'training'})
VAL_DIR   = find_split_dir(DATASET_ROOT, {'validation'})
TEST_DIR  = find_split_dir(DATASET_ROOT, {'evaluation'})

assert all([TRAIN_DIR, VAL_DIR, TEST_DIR]), 'Could not find all split directories!'
print(f'Train : {TRAIN_DIR}')
print(f'Val   : {VAL_DIR}')
print(f'Test  : {TEST_DIR}')

Train : dataset/food11_extracted/training
Val   : dataset/food11_extracted/validation
Test  : dataset/food11_extracted/evaluation


## 3 — Resize to 100×100 and Save
Saves to `dataset/food11_100x100/` on NFS — this is a one-time operation.
`train.ipynb` will then copy from here to local `/tmp` at the start of each session.

In [4]:
NFS_CACHE   = 'dataset/food11_100x100'
TARGET_SIZE = (100, 100)
_resample   = getattr(Image, 'Resampling', Image).LANCZOS

def resize_split(src_dir, dst_dir, size=TARGET_SIZE):
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    exts  = {'.jpg', '.jpeg', '.png'}
    files = [f for f in sorted(src_dir.rglob('*')) if f.suffix.lower() in exts]

    existing = len(list(dst_dir.rglob('*.jpg'))) if dst_dir.exists() else 0
    if existing == len(files):
        print(f'  {src_dir.name:12s} — already done ({existing} files), skipping')
        return

    dst_dir.mkdir(parents=True, exist_ok=True)
    for src in files:
        dst = (dst_dir / src.relative_to(src_dir)).with_suffix('.jpg')
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            Image.open(src).convert('RGB').resize(size, _resample).save(str(dst), 'JPEG', quality=95)
    print(f'  {src_dir.name:12s} — saved {len(files)} images → {dst_dir}')

print(f'Writing pre-resized images to {NFS_CACHE}/ ...')
for src, name in [(TRAIN_DIR, 'training'), (VAL_DIR, 'validation'), (TEST_DIR, 'evaluation')]:
    resize_split(src, Path(NFS_CACHE) / name)
print('Done.')

Writing pre-resized images to dataset/food11_100x100/ ...
  training     — saved 9866 images → dataset/food11_100x100/training
  validation   — saved 3430 images → dataset/food11_100x100/validation
  evaluation   — saved 3347 images → dataset/food11_100x100/evaluation
Done.


In [8]:
# ── Storage speed benchmark: NFS vs /tmp ─────────────────────────────────────
# Reads N images sequentially from each location and reports throughput.
# Run after the copy-to-tmp cell above so both locations are populated.
import time
from pathlib import Path
from PIL import Image

LOCAL_CACHE = '/tmp/ineedfood_100x100'

def _bench(directory, n=500):
    files = list(Path(directory).rglob('*.jpg'))[:n]
    assert len(files) >= n, f'Only {len(files)} files found in {directory}'
    # warm up OS page cache for a fair comparison (read 10 files first)
    for f in files[:10]:
        Image.open(f).load()
    t0 = time.perf_counter()
    for f in files:
        Image.open(f).load()
    elapsed = time.perf_counter() - t0
    return elapsed, len(files) / elapsed   # (seconds, imgs/sec)

N = 500
print(f'Benchmarking sequential read of {N} images ...\n')

nfs_sec, nfs_ips = _bench(str(Path(NFS_CACHE)   / 'training'), N)
tmp_sec, tmp_ips = _bench(str(Path(LOCAL_CACHE)  / 'training'), N)

print(f'  NFS   ({NFS_CACHE}):')
print(f'    {nfs_sec:.2f}s  |  {nfs_ips:.0f} imgs/s')
print(f'  /tmp  ({LOCAL_CACHE}):')
print(f'    {tmp_sec:.2f}s  |  {tmp_ips:.0f} imgs/s')
print(f'\n  Speedup: {tmp_ips/nfs_ips:.1f}x faster on /tmp')
print(f'  Projected full epoch I/O ({len(list(Path(NFS_CACHE+"/training").rglob("*.jpg")))} imgs):')
n_train = len(list((Path(NFS_CACHE) / 'training').rglob('*.jpg')))
print(f'    NFS  : {n_train/nfs_ips:.1f}s')
print(f'    /tmp : {n_train/tmp_ips:.1f}s')

Benchmarking sequential read of 500 images ...

  NFS   (dataset/food11_100x100):
    0.16s  |  3148 imgs/s
  /tmp  (/tmp/ineedfood_100x100):
    0.08s  |  6041 imgs/s

  Speedup: 1.9x faster on /tmp
  Projected full epoch I/O (9866 imgs):
    NFS  : 3.1s
    /tmp : 1.6s


## 4 — Compute Per-Channel Mean & Std

In [11]:
from pathlib import Path

class Food11Dataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root = Path(root_dir)
        self.transform = transform
        self.samples = []
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

        for p in self.root.iterdir():
            if p.is_file() and p.suffix.lower() in exts:
                stem = p.stem
                if "_" not in stem:
                    continue
                cls = int(stem.split("_")[0])   # e.g., 10_200.jpg -> 10
                self.samples.append((p, cls))

        if not self.samples:
            raise RuntimeError(f"No valid images found in {root_dir}")

        self.classes = [str(i) for i in range(11)]
        self.class_to_idx = {c: int(c) for c in self.classes}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label


In [12]:
# Compute stats from the pre-resized training set (no Resize transform needed)
_transform    = T.ToTensor()
_stat_dataset = Food11Dataset(str(Path(NFS_CACHE) / 'training'), transform=_transform)
_stat_loader  = DataLoader(_stat_dataset, batch_size=256, shuffle=False,
                           num_workers=4, pin_memory=False)

mean = torch.zeros(3)
std  = torch.zeros(3)
n    = 0
for imgs, _ in _stat_loader:
    B     = imgs.size(0)
    mean += imgs.mean(dim=[0, 2, 3]) * B
    std  += imgs.std(dim=[0, 2, 3])  * B
    n    += B
mean /= n
std  /= n

MEAN = mean.tolist()
STD  = std.tolist()
print(f'mean : {[f"{v:.4f}" for v in MEAN]}')
print(f'std  : {[f"{v:.4f}" for v in STD]}')

mean : ['0.5544', '0.4508', '0.3440']
std  : ['0.2684', '0.2708', '0.2763']


In [13]:
STATS_FILE = 'dataset/stats.json'
stats = {
    'mean': MEAN,
    'std':  STD,
    'image_size': list(TARGET_SIZE),
    'num_classes': len(_stat_dataset.classes),
    'classes': _stat_dataset.classes,
    'nfs_cache': NFS_CACHE,
    'split_sizes': {
        'training':   len(list((Path(NFS_CACHE) / 'training').rglob('*.jpg'))),
        'validation': len(list((Path(NFS_CACHE) / 'validation').rglob('*.jpg'))),
        'evaluation': len(list((Path(NFS_CACHE) / 'evaluation').rglob('*.jpg'))),
    }
}
with open(STATS_FILE, 'w') as f:
    json.dump(stats, f, indent=2)

print(f'Stats saved to {STATS_FILE}')
print(json.dumps(stats, indent=2))

Stats saved to dataset/stats.json
{
  "mean": [
    0.5544418096542358,
    0.45080119371414185,
    0.3440355658531189
  ],
  "std": [
    0.2684442400932312,
    0.2708316445350647,
    0.27629154920578003
  ],
  "image_size": [
    100,
    100
  ],
  "num_classes": 11,
  "classes": [
    "0",
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10"
  ],
  "nfs_cache": "dataset/food11_100x100",
  "split_sizes": {
    "training": 9866,
    "validation": 3430,
    "evaluation": 3347
  }
}
